# Practical demonstration 2: from super-resolved images to maps

*SSTA workshop, 26 November 2026. Goes with Session 3, "Sample use cases".*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/demo2_use_cases.ipynb)

Session 3's rule: **test super-resolution on a small labelled sample before you use it.**
Here you run that test for the place you worked on this morning.

One **land-cover model** draws all the maps. It was trained on real high-resolution images
only: it has never seen Sentinel-2 or a super-resolved image.

Same three beats as this morning, and no code to write:

* **▶ Look**: run the cell (**Shift + Enter**). Pictures and controls appear under it.
* **⚙ Under the hood** (optional): a collapsed cell with the code and the method. Double-click
  its title to open it.
* **✎ Your turn**: explore with the controls, in pairs.

## 0. Setup

Run the next cell. No key and no GPU needed.

In [ ]:
# @title Helper functions (run this cell; you do not need to read it) { display-mode: "form" }
import json
import os
import re
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle
from scipy import ndimage as ndi

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
INPUTS = ["Sentinel-2", "Stage 1", "Stage 2"]
KEYS = {"Sentinel-2": "s2", "Stage 1": "stage1", "Stage 2": "stage2"}
COLOURS = {"Sentinel-2": "#a6a6a6", "Stage 1": "#7fbfdf", "Stage 2": "#007DB7"}
TASK_CLASS = {"buildings": 7, "roads": 3}
PIXEL_M = 2.5
TOLERANCES = [0, 2.5, 5]   # metres: a pixel counts if the other map has one this close
TOLERANCE_M = 2.5          # the default
METRICS = ["F1", "IoU", "precision", "recall"]
OVERALL = ["F1", "IoU"]   # the metrics that judge a map as a whole
NO_COLOUR = "#d7191c"
OK_COLOUR, FALSE_COLOUR, MISS_COLOUR = (0.10, 0.70, 0.30), (0.20, 0.45, 0.95), (0.90, 0.20, 0.20)


# ---- the data package ---------------------------------------------------------------------

def _fetch(name, url, folder="demo2_data"):
    """A file of the data package: from a local folder, or downloaded once."""
    if Path(url).is_dir():
        return Path(url) / name
    path = Path(folder) / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(f"{url.rstrip('/')}/{name}", path)
    return path


def get_summary(url):
    """Scores of every place (computed before the workshop)."""
    return json.loads(_fetch("summary.json", url).read_text())


class Place:
    """Everything about one place: the three images, the model's map of each, the public
    reference maps and the scores."""

    def __init__(self, name, url):
        slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        z = np.load(_fetch(f"{slug}.npz", url))
        self.name = name
        self.meta = json.loads(str(z["meta"]))
        self.images = {i: z[KEYS[i]].astype("float32") / 10000 for i in INPUTS}   # 4 bands, 0-1
        self.maps = {i: z[f"lc_{KEYS[i]}"] for i in INPUTS}                       # classes 0-7
        self.refs = {"buildings": z["ref_buildings"].astype(bool),
                     "roads": z["ref_roads"].astype(bool)}
        self.valid = z["valid"].astype(bool)
        self.classes = self.meta["classes"]
        self.class_colours = self.meta["class_colours"]
        self.size_m = self.valid.shape[0] * PIXEL_M

    def __repr__(self):
        m = self.meta
        return (f"{self.name}, {m['date']}: {self.size_m / 1000:.0f} km square, "
                f"{m['n_buildings']:,} buildings and {m['n_road_segments']:,} road segments "
                f"in the public maps" +
                (f"; warnings: {', '.join(m['warnings'])}" if m["warnings"] else ""))

    def prediction(self, task, inp):
        """True where the model's map from this input says 'building' (or 'road')."""
        return self.maps[inp] == TASK_CLASS[task]


def _centre(shape, size_m, east_m, north_m):
    """Centre (row, col) and half-width (pixels) of the view, kept inside the area."""
    h, w = shape[-2:]
    half = int(round(size_m / PIXEL_M / 2))
    cy = int(np.clip(h // 2 - north_m / PIXEL_M, half, h - half)) if 2 * half < h else h // 2
    cx = int(np.clip(w // 2 + east_m / PIXEL_M, half, w - half)) if 2 * half < w else w // 2
    return cy, cx, half


def window(arr, size_m=2000, east_m=0, north_m=0):
    """A square of size_m metres cut from an image or map (last two axes are y, x); its
    centre is moved east_m and north_m from the middle of the area."""
    cy, cx, half = _centre(arr.shape, size_m, east_m, north_m)
    return arr[..., max(cy - half, 0):cy + half, max(cx - half, 0):cx + half]


def rgb(img, ceiling=0.3):
    """Natural colour for display from a (4, y, x) reflectance image."""
    return np.clip(np.moveaxis(img[:3], 0, -1) / ceiling, 0, 1)


def _off(axes):
    for ax in np.ravel(axes):
        ax.set_xticks([])
        ax.set_yticks([])


# ---- figures (each returns a matplotlib figure) ------------------------------------------

def fig_locator(place, size_m, east_m, north_m):
    """The whole area, small, with a yellow frame around the current view."""
    cy, cx, half = _centre(place.valid.shape, size_m, east_m, north_m)
    fig, ax = plt.subplots(figsize=(2.4, 2.4))
    ax.imshow(rgb(place.images["Stage 2"][:, ::4, ::4]), interpolation="nearest",
              extent=(0, place.valid.shape[1], place.valid.shape[0], 0))
    ax.add_patch(Rectangle((cx - half, cy - half), 2 * half, 2 * half, fill=False,
                           ec="#ffd400", lw=2))
    ax.set_title("where you are", fontsize=9)
    _off(ax)
    fig.tight_layout()
    return fig


def fig_inputs(place, size_m=1000, east_m=0, north_m=0):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.4))
    for ax, inp in zip(axes, INPUTS):
        ax.imshow(rgb(window(place.images[inp], size_m, east_m, north_m)),
                  interpolation="nearest")
        ax.set_title(inp + (" (10 m, enlarged)" if inp == "Sentinel-2" else " (2.5 m)"))
    _off(axes)
    fig.suptitle(f"{place.name}: {size_m} m across")
    fig.tight_layout()
    return fig


def fig_landcover(place, size_m=1000, east_m=0, north_m=0):
    """Top: the three images. Bottom: the land-cover map the model drew from each."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10.8))
    cmap = ListedColormap(place.class_colours)
    for j, inp in enumerate(INPUTS):
        axes[0, j].imshow(rgb(window(place.images[inp], size_m, east_m, north_m)),
                          interpolation="nearest")
        axes[0, j].set_title(inp)
        axes[1, j].imshow(window(place.maps[inp], size_m, east_m, north_m), cmap=cmap, vmin=0,
                          vmax=len(place.classes) - 1, interpolation="nearest")
        axes[1, j].set_title(f"map from {inp}")
    _off(axes)
    fig.legend([Patch(facecolor=c, edgecolor="grey") for c in place.class_colours],
               place.classes, loc="lower center", ncol=8, fontsize=10, frameon=False)
    fig.suptitle(f"{place.name}: {size_m} m across", y=0.995)
    fig.tight_layout(rect=(0, 0.04, 1, 0.98), h_pad=3.0)
    return fig


def fig_reference(place, size_m=2000, east_m=0, north_m=0):
    """The public maps on top of the Stage 2 image: buildings red, roads yellow."""
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(rgb(window(place.images["Stage 2"], size_m, east_m, north_m)),
              interpolation="nearest")
    roads = window(place.refs["roads"], size_m, east_m, north_m)
    buildings = window(place.refs["buildings"], size_m, east_m, north_m)
    over = np.zeros(roads.shape + (4,))
    over[roads] = (1, 0.85, 0, 0.8)
    over[buildings] = (1, 0.2, 0.2, 0.6)
    ax.imshow(over, interpolation="nearest")
    ax.set_title("the public maps: buildings red, roads yellow")
    _off(ax)
    fig.tight_layout()
    return fig


# ---- scoring -----------------------------------------------------------------------------

def tol_text(metres):
    return "pixel-exact" if metres == 0 else f"within {metres:g} m"


def tol_key(metres):
    return f"{metres:g}"


def near(mask, metres=TOLERANCE_M, pixel_m=PIXEL_M):
    """True wherever ``mask`` has a pixel within ``metres``."""
    r = int(round(metres / pixel_m))
    if r == 0:
        return mask
    y, x = np.ogrid[-r:r + 1, -r:r + 1]
    return ndi.binary_dilation(mask, structure=x * x + y * y <= r * r)


def from_shares(found, correct):
    """F1 and IoU from the two shares. IoU = F1 / (2 - F1): always lower than F1, same order."""
    f1 = 2 * found * correct / max(found + correct, 1e-9)
    return {"F1": f1, "IoU": f1 / (2 - f1)}


def scores(pred, ref, valid, metres=TOLERANCE_M):
    """recall: share of the reference's pixels with a model pixel within ``metres`` (what the
    model found); precision: share of the model's pixels with a reference pixel within
    ``metres`` (what is right of what it marks); F1 and IoU combine the two (0 = nothing
    right, 1 = perfect)."""
    found = np.sum(ref & near(pred, metres) & valid) / max(np.sum(ref & valid), 1)
    correct = np.sum(pred & near(ref, metres) & valid) / max(np.sum(pred & valid), 1)
    return {**from_shares(found, correct), "precision": correct, "recall": found}


def everywhere(place, task, metres=TOLERANCE_M):
    """Scores of a useless map that says 'building' (or 'road') everywhere: the line a real
    map has to beat. In a dense city it is high, because most places are near a building."""
    s = scores(np.ones_like(place.valid), place.refs[task], place.valid, metres)
    return {m: float(s[m]) for m in METRICS}


def score_table(place, metres=TOLERANCE_M):
    """F1, IoU, precision and recall of each input for buildings and roads."""
    rows = []
    for task in TASK_CLASS:
        for inp in INPUTS:
            s = scores(place.prediction(task, inp), place.refs[task], place.valid, metres)
            rows.append({"task": task, "input": inp, **{k: round(float(v), 2)
                                                        for k, v in s.items()}})
    table = pd.DataFrame(rows).set_index(["task", "input"])
    table.attrs["everywhere"] = {t: everywhere(place, t, metres) for t in TASK_CLASS}
    table.attrs["metres"] = metres
    return table


def nothing(metric):
    """Score of a map that says 'no' everywhere: it finds nothing, so F1, IoU and recall are
    0; it marks nothing, so its precision does not exist (None)."""
    return None if metric == "precision" else 0.0


def _bars(ax, values, inputs, base=None, metric="F1", metres=TOLERANCE_M, legend=True,
          no_line=False):
    """Grouped bars: rows of ``values`` = groups on the x axis, columns = inputs.
    ``base``: scores of a map that says yes everywhere (dashed; None = hidden);
    ``no_line``: also show a map that says no everywhere (dotted)."""
    values[inputs].plot.bar(ax=ax, color=[COLOURS[i] for i in inputs], rot=0, width=0.78,
                            legend=False)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", fontsize=7.5)
    if base is not None:
        for k, b in enumerate(base):
            if b is not None:
                ax.hlines(b, k - 0.42, k + 0.42, colors="k", linestyles="--", lw=1.2,
                          label="a map that says yes everywhere" if k == 0 else None)
    if no_line:
        label = "a map that says no everywhere"
        if nothing(metric) is None:     # nothing to draw: say why in the legend
            ax.plot([], [], color=NO_COLOUR, ls=":", lw=2,
                    label=label + " (no precision: it marks nothing)")
        else:
            for k in range(len(values)):
                ax.hlines(nothing(metric), k - 0.42, k + 0.42, colors=NO_COLOUR,
                          linestyles=":", lw=2.5, clip_on=False, zorder=5,
                          label=label if k == 0 else None)
    ax.set_ylabel(f"{metric} (1 = perfect)\n{tol_text(metres)}")
    ax.set_xlabel("")
    ax.set_ylim(0, 1.06)
    if legend:      # below the axes: the dashed line can be anywhere up to 1 (recall)
        ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.1),
                  fontsize=8)


def fig_scores(table, title, metric="F1", yes=True, no=False):
    vals = table[metric].unstack("input")[INPUTS].loc[list(TASK_CLASS)]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    base = [table.attrs["everywhere"][t][metric] for t in vals.index] if yes else None
    _bars(ax, vals, INPUTS, base, metric, table.attrs["metres"], no_line=no)
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    return fig


def best_input(table, task, metric="F1", margin=0.02):
    v = table[metric if metric in OVERALL else "F1"][task]
    best = v.idxmax()
    if v[best] - v["Sentinel-2"] < margin:
        return "no real difference"
    return best


def guess_lines(table, guesses, metric="F1"):
    """Was the guess right? One line per task."""
    lines = []
    for task, guess in guesses.items():
        judge = metric if metric in OVERALL else "F1"   # precision or recall alone cannot
        result = best_input(table, task, judge)          # say which map is best
        if guess in (None, ""):
            mark = "you did not guess"
        else:
            mark = "as you guessed ✓" if guess == result else f"you guessed: {guess}"
        v = table[judge][task]
        base = table.attrs["everywhere"][task][judge]
        lines.append(f"{task:<10} best: {result:<18} ({mark})   {judge} " +
                     ", ".join(f"{i} {v[i]:.2f}" for i in INPUTS) +
                     f";  saying yes everywhere: {base:.2f}")
    return lines


def _error_image(pred, ref, valid, metres=TOLERANCE_M):
    """Green: right (within ``metres`` of the other map). Blue: false alarm (only the
    model). Red: missed (only the public map)."""
    near_pred, near_ref = near(pred, metres), near(ref, metres)
    img = np.full(pred.shape + (3,), 0.12)
    img[ref & ~near_pred] = MISS_COLOUR
    img[pred & ~near_ref] = FALSE_COLOUR
    img[(pred & near_ref) | (ref & near_pred)] = OK_COLOUR
    img[~valid] = 0
    return img


def fig_errors(place, size_m=500, east_m=0, north_m=0, task="buildings", metres=TOLERANCE_M,
               metric="F1"):
    """Top: the three images. Bottom: the model's map of each against the public map.
    Green: correct. Blue: false alarm (only the model). Red: missed (only the public map)."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10.6))
    valid = window(place.valid, size_m, east_m, north_m)
    ref = window(place.refs[task], size_m, east_m, north_m)
    for j, inp in enumerate(INPUTS):
        axes[0, j].imshow(rgb(window(place.images[inp], size_m, east_m, north_m)),
                          interpolation="nearest")
        axes[0, j].set_title(inp)
        pred = window(place.prediction(task, inp), size_m, east_m, north_m)
        axes[1, j].imshow(_error_image(pred, ref, valid, metres), interpolation="nearest")
        here = scores(pred, ref, valid, metres)[metric]
        axes[1, j].set_title(f"{task} from {inp}: {metric} here {here:.2f}")
    _off(axes)
    fig.legend([Patch(color=c) for c in (OK_COLOUR, FALSE_COLOUR, MISS_COLOUR)],
               [f"right ({tol_text(metres)})", "false alarm (only the model)",
                "missed (only the public map)"],
               loc="lower center", ncol=3, frameon=False)
    fig.suptitle(f"{place.name}, {task}: {size_m} m across", y=0.995)
    fig.tight_layout(rect=(0, 0.04, 1, 0.98), h_pad=3.0)
    return fig


def _matrix(ax, m, rows, cols, title, fmt="{:.0%}", vmax=None):
    ax.imshow(m, cmap="Blues", vmin=0, vmax=vmax or max(m.max(), 1e-9))
    for (i, j), v in np.ndenumerate(m):
        ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=9,
                color="white" if v > 0.55 * (vmax or m.max()) else "black")
    ax.set_xticks(range(len(cols)), cols, fontsize=8)
    ax.set_yticks(range(len(rows)), rows, fontsize=8)
    ax.set_title(title, fontsize=10)


# ---- the model at home: OpenEarthMap -------------------------------------------------------

OEM_INPUTS = ["aerial", *INPUTS]
COLOURS["aerial"] = "#3a3a3a"


def get_oem(url):
    """The land-cover model on 79 hand-labelled OpenEarthMap tiles (scores and 3 examples)."""
    oem = json.loads(_fetch("oem_reference.json", url).read_text())
    oem["ex"] = dict(np.load(_fetch("oem_examples.npz", url)))
    return oem


def oem_table(oem, metres=TOLERANCE_M):
    s = oem["scores"][tol_key(metres)]
    rows = [{"task": t, "input": i, **{k: round(v, 2) for k, v in s[t][i].items()}}
            for t in TASK_CLASS for i in OEM_INPUTS]
    table = pd.DataFrame(rows).set_index(["task", "input"])
    table.attrs["everywhere"] = {t: s[t]["everywhere"] for t in TASK_CLASS}
    table.attrs["metres"] = metres
    return table


def fig_oem_scores(oem, metric="F1", metres=TOLERANCE_M, yes=True, no=False):
    table = oem_table(oem, metres)
    vals = table[metric].unstack("input")[OEM_INPUTS].loc[list(TASK_CLASS)]
    fig, ax = plt.subplots(figsize=(8.5, 3.9))
    base = [table.attrs["everywhere"][t][metric] for t in vals.index] if yes else None
    _bars(ax, vals, OEM_INPUTS, base, metric, metres, no_line=no)
    ax.set_title(f"At home: {oem['n_tiles']} hand-labelled OpenEarthMap tiles in "
                 f"{len(oem['cities'])} cities", fontsize=10)
    fig.tight_layout()
    return fig


def fig_oem_confusion(oem, inp="Stage 2"):
    """8-class confusion matrix (pixel-exact): of all pixels of each hand-drawn class, the
    share the model gave to each class. The diagonal is right."""
    cm = np.array(oem["confusion"][inp], dtype=float)
    m = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    names = oem["classes"][1:]
    fig, ax = plt.subplots(figsize=(7.6, 6.4))
    _matrix(ax, m, [f"{n} ({s:.0%})" for n, s in zip(names, cm.sum(1) / cm.sum())],
            names, f"At home, map from {inp}: where each hand-drawn class ends up\n"
            f"(pixel-exact; {np.trace(cm) / cm.sum():.0%} of all pixels right)", vmax=1)
    ax.set_xticklabels(names, rotation=40, ha="right")
    ax.set_ylabel("drawn by hand (share of the area)")
    ax.set_xlabel("the model says")
    fig.tight_layout()
    return fig


def fig_oem_example(oem, i=0):
    """One OpenEarthMap tile: hand-drawn labels, the four images, and the map from each."""
    ex, colours = oem["ex"], oem["class_colours"]
    cmap = ListedColormap(["#000000", *colours])
    fig, axes = plt.subplots(2, 5, figsize=(17, 7.4))
    axes[0, 0].imshow(ex[f"{i}_label"], cmap=cmap, vmin=0, vmax=8, interpolation="nearest")
    axes[0, 0].set_title("drawn by hand (the truth)")
    for j, inp in enumerate(OEM_INPUTS, start=1):
        axes[0, j].imshow(ex[f"{i}_{inp}"], interpolation="nearest")
        axes[0, j].set_title(inp + (" image (0.5 m)" if inp == "aerial" else ""))
        axes[1, j].imshow(ex[f"{i}_map_{inp}"], cmap=cmap, vmin=0, vmax=8,
                          interpolation="nearest")
        axes[1, j].set_title(f"map from {inp}")
    axes[1, 0].axis("off")
    axes[1, 0].legend([Patch(facecolor=c, edgecolor="grey") for c in colours],
                      oem["classes"][1:], loc="center", fontsize=10, frameon=False)
    _off(axes)
    name = str(ex["names"][i])
    fig.suptitle(f"OpenEarthMap tile {name.rsplit('_', 1)[0].replace('_', ' ').title()}, "
                 "500 m across", y=0.995)
    fig.tight_layout(h_pad=2.5)
    return fig


# ---- all places --------------------------------------------------------------------------

def all_places(summary, value="F1", metres=TOLERANCE_M):
    """One table for all places: rows = place and task, columns = inputs."""
    rows = []
    for name, s in summary.items():
        sc = s["scores"][tol_key(metres)]
        for task in TASK_CLASS:
            rows.append({"place": name, "task": task,
                         **{inp: sc[task][inp].get(value) for inp in INPUTS}})
    return pd.DataFrame(rows).set_index(["task", "place"]).sort_index()


def _short(name):
    return (name.split(",")[0].replace(" office", "").replace(" headquarters", " HQ")
            .replace(" City and Mandaue", "").replace(" rice farms", ""))


def fig_all_places(summary, metric="F1", metres=TOLERANCE_M, own_model=False, yes=True,
                   no=False):
    """Bars for every place; dashed: a map that says yes everywhere; dotted: no everywhere."""
    value = f"own_model_{metric}" if own_model else metric
    table = all_places(summary, value, metres)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
    for ax, task in zip(axes, TASK_CLASS):
        t = table.loc[task]
        base = None if own_model or not yes else [
            summary[n]["scores"][tol_key(metres)][task]["everywhere"][metric] for n in t.index]
        t.index = [_short(n) for n in t.index]
        _bars(ax, t, INPUTS, base, metric, metres, legend=False, no_line=no and not own_model)
        ax.set_title(task.capitalize() + (" (a model trained on each input)" if own_model else ""))
        ax.tick_params(axis="x", labelsize=8)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, ncol=5, loc="lower center", fontsize=8)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    return fig


def show_all_places(summary, metric="F1", metres=TOLERANCE_M, yes=True, no=False):
    plt.show(fig_all_places(summary, metric, metres, yes=yes, no=no))
    return all_places(summary, metric, metres)


PLACES = {'ADB headquarters, Mandaluyong': {'lat': 14.5878,
                                   'lon': 121.0585,
                                   'date': '2026-04-15',
                                   'note': 'dense city: office towers, malls, residential '
                                           'blocks'},
 'DHSUD office, Quezon City': {'lat': 14.6447,
                               'lon': 121.0514,
                               'date': '2026-04-15',
                               'note': 'city: government offices, tree-lined streets, housing'},
 'PhilSA office, Quezon City': {'lat': 14.6096,
                                'lon': 121.0803,
                                'date': '2026-04-15',
                                'note': 'city by the Marikina river: towers, subdivisions, '
                                        'riverside'},
 'Cebu City and Mandaue': {'lat': 10.33,
                           'lon': 123.92,
                           'date': '2026-03-20',
                           'note': 'dense city by the coast: port, warehouses, small houses'},
 'Nueva Ecija rice farms': {'lat': 15.5784,
                            'lon': 120.9726,
                            'date': '2026-03-15',
                            'note': 'rice fields, villages and farm-to-market roads'},
 'Bangkok, Thailand': {'lat': 13.75,
                       'lon': 100.4915,
                       'date': '2026-01-15',
                       'note': 'old city on the Chao Phraya: Grand Palace, Sanam Luang, dense '
                               'housing, river'},
 'Phnom Penh, Cambodia': {'lat': 11.565,
                          'lon': 104.925,
                          'date': '2026-02-01',
                          'note': 'city centre: Central Market, Royal Palace, riverfront, '
                                  'dense blocks'},
 'Dhaka, Bangladesh': {'lat': 23.76,
                       'lon': 90.385,
                       'date': '2026-01-15',
                       'note': 'city centre: National Parliament, Farmgate, old airport, dense '
                               'housing'}}
PLACE_DEFAULT = "Cebu City and Mandaue"
DATA_URL = os.environ.get("DEMO2_DATA_URL", "https://huggingface.co/datasets/adbsr-internal/ssta-hackathon/resolve/main/demo2")


# ---- controls shared by the workshop notebooks: answers that save themselves, figures as images
import html as _html
import io

import ipywidgets as W
from IPython.display import display

ANSWERS = {}    # your answers; saved as soon as you type or click
UI = {}         # the controls of each step (used for testing)
_WIDE = W.Layout(width="520px")
_LABEL = {"description_width": "150px"}


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def _note(text, colour="#333"):
    return f"<div style='color:{colour}; line-height:1.5'>{text}</div>"


def _pre(text):
    return f"<pre style='margin:4px 0; line-height:1.35'>{_html.escape(text)}</pre>"


# ---- answers that save themselves ------------------------------------------------------------

def ask(*questions):
    """Each question is (key, text, kind): kind is "text" or a list of choices. The answer
    is saved in ANSWERS the moment it changes; nothing to run."""
    rows = []
    for key, text, kind in questions:
        saved = W.HTML()
        if kind == "text":
            field = W.Text(value=ANSWERS.get(key, ""), placeholder="type here",
                           continuous_update=True, layout=_WIDE)
        else:
            field = W.ToggleButtons(options=list(kind), value=ANSWERS.get(key),
                                    style={"button_width": "auto"})

        def on_change(change, key=key, saved=saved):
            ANSWERS[key] = change["new"]
            saved.value = _note("saved ✓", "#1a7f37") if change["new"] not in ("", None) else ""
        field.observe(on_change, names="value")
        rows.append(W.VBox([W.HTML(f"<b>{text}</b>"), W.HBox([field, saved])],
                           layout=W.Layout(margin="6px 0 10px 0")))
        rows[-1].field = field
    box = W.VBox(rows, layout=W.Layout(border="1px solid #007DB7", padding="8px 12px",
                                       margin="8px 0"))
    box.fields = [r.field for r in rows]
    display(box)
    return box


# ---- "under the hood" cells: the plain code behind a part ------------------------------------

class NotYet(Exception):
    """An earlier part is missing. Shown as one line, without a traceback."""

    def _render_traceback_(self):
        return [f"\x1b[31m{self}\x1b[0m"]


def explain(key):
    """The note that goes with an "under the hood" cell (the notebook's NOTES)."""
    from IPython.display import Markdown
    display(Markdown(NOTES[key].strip("\n")))


# ---- the notebook's controls: menus, buttons and sliders that act at once ---------------------
S = {}          # this session: the place that is loaded, the scores
REFRESH = []    # what to redraw when the place changes
SCORE_REFRESH = []   # what to redraw when the tolerance or the metric changes
SCORING = {"metres": TOLERANCE_M, "metric": "F1", "yes": True, "no": False}
BEST = ["Sentinel-2", "Stage 1", "Stage 2", "no real difference"]
_SWITCHES = {"metres": [], "metric": [], "yes": [], "no": []}
_BUSY = [False]


def _place_needed():
    if "place" not in S:
        display(W.HTML(_note("<b>Run the cell of part 2 first</b>, then run this cell again.",
                             "#b00020")))
        return False
    return True


def _refresh_all():
    for redraw in REFRESH:
        redraw()


# ---- the scoring switches: every copy shows the same value, and every score follows ---------

def _on_switch(change, key):
    if _BUSY[0]:
        return
    _BUSY[0] = True
    try:
        SCORING[key] = change["new"]
        for w in _SWITCHES[key]:
            w.value = change["new"]          # the other copies follow
    finally:
        _BUSY[0] = False
    for redraw in SCORE_REFRESH:
        redraw()


def _switch(key):
    if key == "metres":
        w = W.ToggleButtons(options=[("pixel-exact", 0), ("within 2.5 m", 2.5),
                                     ("within 5 m", 5)], value=SCORING["metres"],
                            description="A pixel counts", style={"button_width": "105px",
                                                                 "description_width": "150px"},
                            tooltips=["the two maps must agree on the very pixel",
                                      "one pixel of shift is forgiven",
                                      "two pixels of shift are forgiven"])
    elif key in ("yes", "no"):
        w = W.Checkbox(value=SCORING[key], indent=False, layout=W.Layout(width="250px"),
                       description=f"show a map that says {key} everywhere")
    else:
        w = W.ToggleButtons(options=METRICS, value=SCORING["metric"], description="Score",
                            style={"button_width": "80px", "description_width": "150px"},
                            tooltips=["precision and recall combined",
                                      "intersection over union: lower than F1, same order",
                                      "of what the model marks, the share that is right",
                                      "of what the public map has, the share the model found"])
    w.observe(lambda change: _on_switch(change, key), names="value")
    _SWITCHES[key].append(w)
    return w


def _switches(metric=True, lines=True):
    rows = [_switch("metres")] + ([_switch("metric")] if metric else [])
    if lines:
        rows.append(W.HBox([W.Label("Compare with", layout=W.Layout(width="150px")),
                            _switch("yes"), _switch("no")]))
    return W.VBox(rows)


def _lines():
    return {"yes": SCORING["yes"], "no": SCORING["no"]}


# ---- a viewer: move and zoom, the picture follows -------------------------------------------

class Viewer:
    """A figure of the current place with sliders to move the view and buttons to zoom.
    ``render(place, size_m, east_m, north_m, *extra)`` returns the figure; ``extra`` are
    more controls (for example buildings or roads) whose values are passed on."""

    def __init__(self, render, size_m=1000, extra=()):
        self.render, self.extra = render, list(extra)
        self.size = W.ToggleButtons(options=[("250 m", 250), ("500 m", 500), ("1 km", 1000),
                                             ("whole area", 2000)], value=size_m,
                                    description="Zoom", style={"button_width": "90px",
                                                               "description_width": "150px"})
        self.east = W.IntSlider(value=0, min=-750, max=750, step=50, continuous_update=False,
                                description="← west | east →", layout=_WIDE, style=_LABEL)
        self.north = W.IntSlider(value=0, min=-750, max=750, step=50, continuous_update=False,
                                 description="← south | north →", layout=_WIDE, style=_LABEL)
        self.locator = W.Image(format="png", layout=W.Layout(width="190px"))
        self.image = W.Image(format="png")
        for w in (self.size, self.east, self.north, *self.extra):
            w.observe(self.draw, names="value")
        controls = W.VBox([*self.extra, self.size, self.east, self.north])
        self.box = W.VBox([W.HBox([controls, self.locator]), self.image])
        REFRESH.append(self.draw)
        self.draw()

    def draw(self, _=None):
        place = S["place"]
        args = (self.size.value, self.east.value, self.north.value)
        self.locator.value = _png(fig_locator(place, *args))
        self.image.value = _png(self.render(place, *args, *[w.value for w in self.extra]))


def _task_buttons(value="buildings"):
    return W.ToggleButtons(options=list(TASK_CLASS), value=value, description="Map of",
                           style={"button_width": "110px", "description_width": "150px"})


# ---- part 1: the model at home: pictures first, then numbers ----------------------------------

def _oem(url):
    """The OpenEarthMap results, loaded once; None (with a message) if they cannot be loaded."""
    if "oem" not in S:
        try:
            S["oem"] = get_oem(url)
        except Exception as exc:  # no network, wrong address: say so, do not crash
            display(W.HTML(_note(f"Could not load the data: {_html.escape(str(exc))}",
                                 "#b00020")))
            return None
    return S["oem"]


def home_examples(url):
    """Three hand-labelled OpenEarthMap tiles: the four images and the model's map of each."""
    oem = _oem(url)
    if oem is None:
        return
    tile = W.ToggleButtons(options=[(f"example {i + 1}: " +
                                     str(n).rsplit("_", 1)[0].replace("_", " ").title(), i)
                                    for i, n in enumerate(oem["ex"]["names"])],
                           value=0, description="Look at", style={"button_width": "190px",
                                                                  "description_width": "150px"})
    example = W.Image(format="png")

    def draw_example(_=None):
        example.value = _png(fig_oem_example(oem, tile.value))
    tile.observe(draw_example, names="value")
    draw_example()
    ui = W.VBox([tile, example])
    ui.tile = tile
    UI["examples"] = ui
    display(ui)


def home_scores(url):
    """The scores over all OpenEarthMap tiles, and which classes the model confuses."""
    oem = _oem(url)
    if oem is None:
        return
    chart = W.Image(format="png", layout=W.Layout(width="720px"))
    table = W.HTML()
    which = W.ToggleButtons(options=OEM_INPUTS, value="Stage 2", description="Map from",
                            style={"button_width": "100px", "description_width": "150px"})
    matrix = W.Image(format="png", layout=W.Layout(width="560px"))

    def draw_matrix(_=None):
        matrix.value = _png(fig_oem_confusion(oem, which.value))
    which.observe(draw_matrix, names="value")

    def draw_scores(_=None):
        m, metric = SCORING["metres"], SCORING["metric"]
        chart.value = _png(fig_oem_scores(oem, metric, m, **_lines()))
        tab = oem_table(oem, m).copy()
        tab.index = [f"{t}, {i}" for t, i in tab.index]
        table.value = tab.to_html(float_format="%.2f")
    SCORE_REFRESH.append(draw_scores)
    draw_scores()
    draw_matrix()
    ui = W.VBox([_switches(), chart, table,
                 W.HTML("<br><b>Which classes does the model confuse?</b> (pixel-exact)"),
                 which, matrix])
    ui.which, ui.matrix = which, matrix
    UI["home"] = ui
    display(ui)


# ---- step 2: pick your place ----------------------------------------------------------------

def pick_place(url):
    """A menu of the places: choosing one loads it, and every part below follows."""
    menu = W.Dropdown(options=list(PLACES), value=PLACE_DEFAULT, description="Place",
                      layout=_WIDE, style=_LABEL)
    status = W.HTML()
    holder = W.VBox([])

    def load(_=None):
        menu.disabled = True
        status.value = _note(f"Loading {_html.escape(menu.value)} ...")
        try:
            S["place"] = Place(menu.value, url)
            ANSWERS["PLACE"] = menu.value
            status.value = _note(_html.escape(repr(S["place"])), "#555")
            if holder.children:
                _refresh_all()          # every step below follows the new place
            else:
                holder.children = (Viewer(fig_inputs, size_m=1000).box,)
        except Exception as exc:  # no network, wrong address: say so, do not crash
            status.value = _note(f"Could not load the data: {_html.escape(str(exc))}",
                                 "#b00020")
        finally:
            menu.disabled = False
    menu.observe(load, names="value")
    ui = W.VBox([menu, status, holder])
    ui.menu, ui.status = menu, status
    UI["place"] = ui
    display(ui)
    load()


# ---- step 3: guess --------------------------------------------------------------------------

def guess():
    UI["guess"] = ask(("GUESS_BUILDINGS", "Which image will give the best map of buildings?",
                       BEST),
                      ("GUESS_ROADS", "And the best map of roads?", BEST))


# ---- step 4: the maps -----------------------------------------------------------------------

def show_maps():
    if not _place_needed():
        return
    viewer = Viewer(fig_landcover, size_m=1000)
    UI["maps"] = viewer
    display(viewer.box)


# ---- step 5: the scores ---------------------------------------------------------------------

def show_scores():
    """A button that shows the scores (after you have guessed); the switches change how they
    are counted."""
    if not _place_needed():
        return
    show = W.Button(description="Show the scores", button_style="primary", icon="eye",
                    layout=W.Layout(width="220px"))
    out = W.VBox([])

    def draw(_=None):
        if not show.disabled:
            return                      # not revealed yet
        p, m, metric = S["place"], SCORING["metres"], SCORING["metric"]
        table = score_table(p, m)
        S["table"] = table
        guesses = {"buildings": ANSWERS.get("GUESS_BUILDINGS"),
                   "roads": ANSWERS.get("GUESS_ROADS")}
        tab = table.copy()
        tab.index = [f"{t}, {i}" for t, i in tab.index]
        out.children = (
            W.HBox([W.Image(value=_png(fig_reference(p)), format="png",
                            layout=W.Layout(width="400px")),
                    W.Image(value=_png(fig_scores(table, f"{p.name}: the land-cover model "
                                                         "against the public maps", metric,
                                                  **_lines())),
                            format="png", layout=W.Layout(width="560px"))]),
            W.HTML(_pre("\n".join(guess_lines(table, guesses, metric)))),
            W.HTML(tab.to_html(float_format="%.2f")))

    def reveal(_):
        if not (ANSWERS.get("GUESS_BUILDINGS") and ANSWERS.get("GUESS_ROADS")):
            note.value = _note("Make both guesses in part 3 first (or press again to see the "
                               "scores without a guess).", "#b00020")
            if not note.warned:
                note.warned = True
                return
        note.value = ""
        show.disabled = True
        show.description = "Scores shown"
        draw()
    note = W.HTML()
    note.warned = False
    show.on_click(reveal)
    REFRESH.append(draw)
    SCORE_REFRESH.append(draw)
    ui = W.VBox([_switches(), show, note, out])
    ui.show = show
    UI["scores"] = ui
    display(ui)


# ---- step 6: hunt ---------------------------------------------------------------------------

def hunt():
    if not _place_needed():
        return
    viewer = Viewer(fig_errors, size_m=500,
                    extra=[_task_buttons(), _switch("metres"), _switch("metric")])
    UI["hunt"] = viewer
    display(viewer.box)
    UI["finds"] = ask(("RESCUE", "A rescue: something Sentinel-2 missed and super-resolution "
                       "found", "text"),
                      ("FAILURE", "A failure: something the super-resolved map gets wrong",
                       "text"))


# ---- the result card ------------------------------------------------------------------------

def card_text():
    def get(key):
        v = ANSWERS.get(key)
        return "(not answered)" if v in ("", None) else v

    m = SCORING["metres"]
    lines = ["=" * 68, f"PLACE     {get('PLACE')}", f"SCORED    {tol_text(m)}"]
    if "place" in S:
        table = score_table(S["place"], m)
        for metric in METRICS:
            for t in TASK_CLASS:
                v = table[metric][t]
                base = table.attrs["everywhere"][t][metric]
                lines.append(f"{metric:<9} {t:<10} " + ", ".join(f"{i} {v[i]:.2f}" for i in INPUTS)
                             + f"  (everywhere {base:.2f})")
    lines += ["-" * 68,
              f"my guess, buildings:     {get('GUESS_BUILDINGS')}",
              f"my guess, roads:         {get('GUESS_ROADS')}",
              f"rescued by SR:           {get('RESCUE')}",
              f"failure:                 {get('FAILURE')}",
              "=" * 68]
    return "\n".join(lines)


def result_card():
    card = W.HTML(_pre(card_text()))
    again = W.Button(description="Update the card", icon="refresh")
    again.on_click(lambda _: setattr(card, "value", _pre(card_text())))
    ui = W.VBox([card, again])
    ui.card, ui.again = card, again
    UI["card"] = ui
    display(ui)


# ---- for the discussion: all places, and a model trained on each input ----------------------

def all_places_view(url, own_model=False):
    """Bars and a table for all places; the switches change how they are counted."""
    summary = get_summary(url)
    chart = W.Image(format="png")
    table = W.HTML()

    def draw(_=None):
        m, metric = SCORING["metres"], SCORING["metric"]
        chart.value = _png(fig_all_places(summary, metric, m, own_model,
                                          **({} if own_model else _lines())))
        value = f"own_model_{metric}" if own_model else metric
        table.value = all_places(summary, value, m).to_html(float_format="%.2f")
    SCORE_REFRESH.append(draw)
    draw()
    ui = W.VBox([_switches(lines=not own_model), chart, table])
    UI["own" if own_model else "all"] = ui
    display(ui)


# ---- "under the hood" cells: the plain code behind each part ---------------------------------

def picked_place():
    """The place chosen in the menu of part 2."""
    if "place" not in S:
        raise NotYet("Run the cell of part 2 first (it loads your place), then run this cell "
                     "again.")
    return S["place"]


NOTES = {
    "scores": """
**How the scores work** (the same in the whole notebook)

* **Recall**: the share of the true buildings (or roads) that the model found.
  **Precision**: of what the model marks as building, the share that is right. Each alone can
  be gamed: a map that says "building" everywhere has a perfect recall, and a map that marks
  one sure building has a perfect precision.
* **F1** combines the two: 0 = nothing right, 1 = perfect. **IoU** (intersection over union)
  is the share both maps agree on, out of everything either map calls a building. It is
  always lower than F1 (IoU = F1 / (2 − F1)) and ranks the images in the same order; it is the
  number most papers report.
* **A pixel counts**: two maps never line up to the metre. *Pixel-exact* counts only the very
  same pixel; *within 2.5 m* forgives a shift of one pixel; *within 5 m* forgives two. The
  more you forgive, the higher every score.
* **Two useless maps to compare with** (tick them on or off):
  * a map that says **"building" everywhere** (dashed line, "everywhere" in the table above).
    In a dense city it scores high, because almost every spot is near a building.
  * a map that says **"no building" anywhere** (dotted line). It scores 0 on F1, IoU and
    recall, and has no precision at all: it marks nothing. Yet 83 % of its pixels are right
    here, and 95 % in Nueva Ecija, because most pixels are not buildings. That is why maps of
    rare things are not scored by the share of pixels right.
* **The confusion matrix** (under the examples) shows, for each class drawn by hand, which
  class the model gave those pixels. The diagonal is right; everything else is a confusion.
  It counts pixel-exact: a pixel has to land in one box, so the tolerance does not apply.
""",
    "package": """
**What was done before the workshop**, for each place:

1. **The images** come from the super-resolution service: the same job as this morning
   (`sr.submit(area, date)`). Stage 1 and Stage 2 are its two results. "Sentinel-2" is its
   10 m input (`lr_reference.tif`), enlarged to the same 2.5 m grid, so all three images
   line up pixel by pixel.
2. **The maps**: the land-cover model was run on each of the three images (part 4).
3. **The public maps** were downloaded for the same square and drawn on the same grid, as
   yes/no maps of buildings and of roads (part 5). Above: buildings red, roads yellow.
4. **Valid** marks where the images have data; empty squares are left out of every score.

The notebook only downloads the result (about 12 MB per place), so it needs no key and no
GPU.
""",
    "model": """
**How the maps were made.** The same land-cover model, unchanged, read each of the three
images: red, green and blue at 2.5 m, cut into overlapping squares of 960 m. For every pixel
it gives a probability for each of the 8 classes; the map keeps the most likely one. Roughly,
in code (run before the workshop, not here):

```python
from satlaspretrain_models.model import Model
from satlaspretrain_models.utils import Backbone, Head

model = Model(num_channels=3, multi_image=False, backbone=Backbone.SWINB, fpn=True,
              head=Head.SEGMENT, num_categories=8, weights=None)
model.load_state_dict(weights)                # fine-tuned on OpenEarthMap; on Hugging Face

for name, image in images.items():            # Sentinel-2 enlarged, Stage 1, Stage 2
    probabilities = model(image[:3])[0].softmax(1)
    maps[name] = probabilities.argmax(1)      # one class (0-7) per pixel
```

Nothing was trained or tuned on these images. If the three maps differ, it is only because
the images differ.
""",
    "public": """
**The public maps, and their limits**

* **Open Buildings**: building footprints from Google, Microsoft and OpenStreetMap, combined
  by VIDA. They were made by a model from 50 cm images, so they are good but not perfect.
* **OpenStreetMap roads** (through Overture Maps). Roads are lines, so each one is widened
  to a typical width for its class: 12 m for a motorway, 5 m for a residential street, 3 m
  for a farm track.
* Both are drawn on the 2.5 m grid of the images. An error in the public map counts against
  the model: a road that is missing in OpenStreetMap turns a correct road into a "false
  alarm". So even a perfect model would not reach 1 here.

**The dashed line** is the last row above: a map that says yes everywhere. Its recall is 1 and
its precision is the share of the area that is near a building, which is high in a dense city.
""",
}


FOUND = {'found_home': '\n'
               '**What it shows** (within 2.5 m):\n'
               '\n'
               '* From the **aerial** image the model is very good: F1 0.88 for buildings, '
               '0.76 for roads.\n'
               '  This is what it was built for.\n'
               '* From **Sentinel-2** it is close to useless: 0.35 for buildings and 0.23 for '
               'roads, below\n'
               '  the dashed line (0.40 and 0.25).\n'
               '* **Stage 1** (0.55 and 0.46) and **Stage 2** (0.66 and 0.54) are in between. '
               'Stage 2 closes\n'
               '  a little more than half of the gap between Sentinel-2 and the aerial image, '
               'for buildings\n'
               '  and for roads (half pixel-exact, about two thirds within 5 m).\n'
               '* **Precision and recall** show where the gain comes from. From Sentinel-2 the '
               'model marks\n'
               '  few buildings, and most of those are right (precision 0.68), but it finds '
               'only 23 % of them\n'
               '  (recall). Stage 2 keeps the precision (0.70) and finds 63 %.\n'
               '* **The confusion matrix** tells the same story. From the aerial image, 78 % '
               'of the building\n'
               '  pixels are called building. From Sentinel-2 only 18 % are: the rest become '
               'rangeland\n'
               '  (29 %), developed space (27 %), tree (15 %) and even water (9 %). From Stage '
               '2, 48 %.\n'
               '* The three examples are a good, a typical and a poor tile for Stage 2, not '
               'the best ones.\n'
               '  Look at what Stage 2 gets right (which blocks are built up, the main roads) '
               'and what it\n'
               '  does not (the exact shape of each building, narrow paths).\n'
               '\n'
               'So the model works on super-resolved images, but not as well as on a real '
               'aerial image. Keep\n'
               'that in mind as you leave home: do not expect it to do better at your place '
               'than here.\n',
 'found_places': '\n'
                 '**What we found in the five Philippine places** (our run before the '
                 'workshop; scores within\n'
                 '2.5 m unless said otherwise):\n'
                 '\n'
                 '* **Buildings**: the score rises with every step, in every place: F1 0.21 to '
                 '0.32 on\n'
                 '  Sentinel-2, 0.48 to 0.56 on Stage 1, 0.65 to 0.75 on Stage 2 (pixel-exact '
                 '0.53 to 0.60;\n'
                 '  within 5 m 0.74 to 0.84). Stage 2 finds 64 to 78 % of the buildings '
                 '(recall), and 67 to 79 %\n'
                 '  of what it marks is right (precision). Almost all the gain is recall: on '
                 'Sentinel-2 the\n'
                 '  model already marks few buildings, and those are mostly right (precision '
                 '0.61 to 0.82), but\n'
                 '  it finds only 12 to 20 % of them. But in the four cities a map that says '
                 '"building" everywhere already\n'
                 '  scores 0.54 to 0.72: Stage 2 is above that line by only 0.02 to 0.11. '
                 'Where almost\n'
                 '  everything is built up, the map adds little to "it is all built up". In '
                 'Nueva Ecija, with\n'
                 '  few buildings, the line is at 0.16 and Stage 2 (0.74) is far above it.\n'
                 '* **Roads**: on Sentinel-2 the model finds almost no roads in four of the '
                 'five places (F1\n'
                 '  0.01 to 0.09); only around ADB headquarters, where the avenues are wide, '
                 'does it reach\n'
                 '  0.25. Stage 2 reaches 0.29 to 0.61 and is clearly above the "roads '
                 'everywhere" line (0.30\n'
                 '  to 0.36 in the cities, 0.09 in Nueva Ecija) in every place except Cebu '
                 '(0.29 against 0.30),\n'
                 '  where the streets between small houses are too narrow. Stage 2 finds 25 to '
                 '54 % of the\n'
                 '  roads (recall), and 36 to 71 % of what it marks is right (precision): it '
                 'misses roads more\n'
                 '  often than it invents them. The largest gain is for the farm roads of '
                 'Nueva Ecija (0.01 to 0.61).\n'
                 '* **The tolerance changes the level, not the order.** Forgive more shift and '
                 'every score\n'
                 '  rises, the dashed line too. The order of the three images never changes, '
                 'and the lead of\n'
                 '  Stage 2 over the line moves by a few hundredths (Cebu roads go from just '
                 'above the line,\n'
                 '  pixel-exact, to just below it). IoU gives lower numbers than F1 and the '
                 'same order.\n'
                 '* **At home and away.** Here Stage 2 reaches F1 0.65 to 0.75 for buildings '
                 'and 0.29 to 0.61\n'
                 '  for roads (within 2.5 m); at home it reached 0.66 and 0.54. About the '
                 'same, in spite of the\n'
                 '  new country and the imperfect public maps, but the two are not strictly '
                 'comparable (other\n'
                 '  labels, other places). And the dashed line is far higher in the dense '
                 'cities here (0.54 to\n'
                 '  0.72 for buildings) than at home (0.40), so the same F1 is worth less '
                 'here.\n'
                 '\n'
                 '**Bangkok, Phnom Penh and Dhaka** (Sentinel-2 / Stage 1 / Stage 2, within '
                 '2.5 m) show the same\n'
                 'order:\n'
                 '\n'
                 '* **Buildings**: Bangkok 0.32 / 0.62 / 0.77, Phnom Penh 0.46 / 0.82 / 0.87, '
                 'Dhaka 0.03 / 0.07 /\n'
                 '  0.55. In Dhaka the map from Stage 2 stays below the "building everywhere" '
                 'line (0.60): the\n'
                 '  blocks are very dense, and the model finds only 41 % of the buildings.\n'
                 '* **Roads**: Bangkok 0.08 / 0.44 / 0.53, Phnom Penh 0.04 / 0.46 / 0.59, '
                 'Dhaka 0.02 / 0.13 /\n'
                 '  0.43, all above the "roads everywhere" line (0.30 to 0.35) with Stage 2.\n'
                 '\n'
                 'This is the pattern of Session 3: a model trained on real high-resolution '
                 'images cannot read\n'
                 'a 10 m image, and super-resolution lets it work on free Sentinel-2. In a '
                 'dense city, check\n'
                 'the map against the dashed line before you trust it.\n',
 'found_own': '\n'
              'With a model trained on each image (within 2.5 m): buildings 0.63 to 0.80 on '
              'Sentinel-2\n'
              'against 0.62 to 0.78 on Stage 2; roads 0.25 to 0.37 against 0.24 to 0.38. This '
              'quick model\n'
              'only looks at colours and textures around each pixel and is scored on the east '
              'half only, so\n'
              'it is a rough test. Even so, trained on Sentinel-2 alone it reaches about the '
              'level the\n'
              'land-cover model reached on Stage 2 for buildings.\n',
 'finetune': '\n'
             '**What you need to fine-tune**\n'
             '\n'
             '* **Labels for your area.** Public maps (Open Buildings, OpenStreetMap) are a '
             'start, but you\n'
             '  saw their errors; a few hundred tiles checked or drawn by people who know the '
             'place are\n'
             '  worth more.\n'
             '* **The images you will use.** Fine-tune on the same kind of image you will run '
             'the model on:\n'
             '  if you will map from Stage 2, fine-tune on Stage 2.\n'
             '* **A test area the model never sees during training**, to score it honestly, as '
             'in part 5.\n'
             '* **A GPU for a few hours** (a free Colab GPU is enough for a first try).\n'
             '\n'
             'In code it looks roughly like this (a sketch, not run here):\n'
             '\n'
             '```python\n'
             'import torch\n'
             'from satlaspretrain_models.model import Model\n'
             'from satlaspretrain_models.utils import Backbone, Head\n'
             '\n'
             'model = Model(num_channels=3, multi_image=False, backbone=Backbone.SWINB, '
             'fpn=True,\n'
             '              head=Head.SEGMENT, num_categories=8, weights=None)\n'
             'state = torch.load("landcover_oem_satlas_swinb_fp16.pth")   # from Hugging Face, '
             'see below\n'
             'model.load_state_dict({k: v.float() for k, v in state.items()})\n'
             '\n'
             '# keep what it knows: the backbone learns slowly, the rest faster\n'
             'backbone = list(model.backbone.parameters())\n'
             'rest = [p for n, p in model.named_parameters() if not '
             'n.startswith("backbone.")]\n'
             'opt = torch.optim.AdamW([{"params": backbone, "lr": 1e-5}, {"params": rest, '
             '"lr": 1e-4}],\n'
             '                        weight_decay=1e-4)\n'
             'model.train()\n'
             'for images, labels in tiles_of_your_area:     # (B, 3, H, W) images; (B, H, W) '
             'classes 0-7\n'
             '    _, loss = model(images, labels)\n'
             '    opt.zero_grad()\n'
             '    loss.backward()\n'
             '    opt.step()\n'
             '```\n'}
NOTES.update(FOUND)

## 1. The model at home *(8 minutes)*

**▶ Look.** First the model on home ground: tiles of 500 m, labelled by hand, from cities it
was not trained on. Top row: the hand-drawn labels, the **aerial** image the model knows, and
the three versions you will get for your place. Bottom row: the model's map from each.

In [ ]:
home_examples(DATA_URL)

**✎ Your turn.** Click through the three examples. What does the map from Stage 2 get right,
and what does it miss?

**▶ Look.** Now in numbers, over all 79 tiles. **F1** runs from 0 (nothing right) to 1
(perfect). The **dashed line** is a useless map that says "building" everywhere: a map is
only worth something clearly above it.

In [ ]:
home_scores(DATA_URL)

In [ ]:
# @title Under the hood: how a score is counted { display-mode: "form" }
oem = get_oem(DATA_URL)                       # the scores of part 1, computed before the workshop
print(f"{oem['n_tiles']} tiles in {len(oem['cities'])} cities, labels drawn at "
      f"{oem['gsd_label_m']} m: {', '.join(oem['cities'])}\n")

counted = oem["scores"]["2.5"]["buildings"]   # try "0" or "5" (metres), and "roads"
print(f"{'map from':<12}{'recall':>8}{'precision':>11}{'F1':>7}{'IoU':>7}")
for name in ["aerial", "Sentinel-2", "Stage 1", "Stage 2", "everywhere"]:
    recall = counted[name]["recall"]          # share of the true buildings the model found
    precision = counted[name]["precision"]    # share of what the model marks that is right
    f1 = 2 * precision * recall / (precision + recall)
    iou = f1 / (2 - f1)
    print(f"{name:<12}{recall:>8.2f}{precision:>11.2f}{f1:>7.2f}{iou:>7.2f}")

explain("scores")

**✎ Your turn.** Try the switches: they change every score in this notebook. In the
confusion matrix, compare the aerial image with Sentinel-2: where do the buildings go?

In [ ]:
# @title What we found at home (open after your turn) { display-mode: "form" }
explain("found_home")

## 2. Your place *(5 minutes)*

**▶ Look.** Now the model leaves home. Choose the place you used this morning, or the one
you know best. You get three
images of the same 2 km square: **Sentinel-2** (enlarged), **Stage 1** and **Stage 2**.

In [ ]:
pick_place(DATA_URL)

In [ ]:
# @title Under the hood: what the data of your place holds { display-mode: "form" }
place = picked_place()                        # the place chosen in the menu above
print(place, "\n")

for name in INPUTS:
    image, model_map = place.images[name], place.maps[name]
    print(f"{name:<11} image {image.shape} (4 bands, reflectance 0-1);  "
          f"the model's map {model_map.shape} (one class per pixel)")
print("\nclasses:", dict(enumerate(place.classes)))
for task, public_map in place.refs.items():
    print(f"public map of {task}: {public_map[place.valid].mean():.0%} of the area")

fig_reference(place)                          # the public maps on the Stage 2 image
plt.show()

explain("package")

**✎ Your turn.** Move to a spot you know with the sliders. If you change the place later,
everything below follows.

## 3. Guess first *(2 minutes)*

**✎ Your turn.** Your place is far from the model's home. Which image will give the best
map of **buildings**, and of **roads**? Click your guess before you go on.

In [ ]:
guess()

## 4. The maps *(5 minutes)*

**▶ Look.** Top row: the three images. Bottom row: the map the same model drew from each.
Nobody labelled or trained anything on these images.

In [ ]:
show_maps()

In [ ]:
# @title Under the hood: how the maps were made { display-mode: "form" }
place = picked_place()

# how much of the area the model gives to each class, from each image (%)
valid = place.valid                           # where the images have data
shares = pd.DataFrame({name: np.bincount(place.maps[name][valid], minlength=8) / valid.sum()
                       for name in INPUTS}, index=place.classes)
print((100 * shares).round(0).astype(int).to_string())

explain("model")

**✎ Your turn.** Where do the three maps differ most? What does the map from Sentinel-2 make
of the buildings?

## 5. Hunt: one rescue, one failure *(8 minutes)*

**▶ Look.** Now check the maps against two public maps: **Open Buildings** and
**OpenStreetMap roads**. **Green** is right, **blue** is a false alarm (only the model says
"building"), **red** is missed (only the public map says so).

In [ ]:
hunt()

**✎ Your turn.** Find a **rescue** (Sentinel-2 missed it, super-resolution found it) and a
**failure** (the super-resolved map is wrong; or is the public map wrong?). Try roads too.
Write both in the blue box.

## 6. The scores *(8 minutes)*

**▶ Look.** The same comparison in numbers, counted as in part 1. Press **Show the scores**
to see them, and whether your guess was right.

In [ ]:
show_scores()

In [ ]:
# @title Under the hood: the score of your place, counted in code { display-mode: "form" }
place = picked_place()
task, metres = "buildings", 2.5               # try "roads", and 0 or 5 metres

truth = place.refs[task]                      # the public map: True where it has a building
valid = place.valid                           # where the images have data

print(f"{task}, {tol_text(metres)}\n{'map from':<22}{'recall':>8}{'precision':>11}{'F1':>7}")
maps = {name: place.prediction(task, name) for name in INPUTS}   # True where the model says so
maps['"yes" everywhere'] = np.ones_like(valid)                    # the dashed line
for name, marked in maps.items():
    # near(x, metres): True wherever x has a pixel within that distance
    recall = (truth & near(marked, metres) & valid).sum() / max((truth & valid).sum(), 1)
    precision = (marked & near(truth, metres) & valid).sum() / max((marked & valid).sum(), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    print(f"{name:<22}{recall:>8.2f}{precision:>11.2f}{f1:>7.2f}")

explain("public")

**✎ Your turn.** Compare the three images with each other and with the dashed line, not with
1: the public maps have errors too. Try the tolerances: what changes, what stays? Does
super-resolution help the model *find* more (recall) or make *fewer false alarms*
(precision)?

**Your result card.** Run the next cell and enter your guess and your finds in the poll (the
link is on the slide).

In [ ]:
result_card()

---
## For the discussion: all places

Where does super-resolution help most? Where does it make no difference?

In [ ]:
all_places_view(DATA_URL)

In [ ]:
# @title What we found in the places { display-mode: "form" }
explain("found_places")

## 7. Going further: fine-tune the model

We used the model **as it is**. The usual next step is to **fine-tune** it on labelled
examples from your own area.

**One trap: compare fairly.** Before the workshop we trained a quick model on each of the
three images. Look at the chart: **the gap between the images almost disappears.** A model
trained on Sentinel-2 learns to read Sentinel-2. So:

* if a good model trained on high-resolution images exists for your task, super-resolution
  lets you run it on free Sentinel-2;
* if you train your own model anyway, test whether super-resolution still helps.

Tomorrow's hackathon starts here.

In [ ]:
all_places_view(DATA_URL, own_model=True)

In [ ]:
# @title What we found with a model trained on each image { display-mode: "form" }
explain("found_own")

In [ ]:
# @title How to fine-tune: what you need, and a code sketch { display-mode: "form" }
explain("finetune")

**Before you use a map**

* **Check the warnings and the validity map** of every job. Near water, use Sentinel-2.
* **Spot-check** the map against a real image or a field visit before acting on it.
* **Use maps for areas, not single objects**: screening, planning, trends.
* **Each map shows a period** (45 days before and after the date), not a single day.

The land-cover model is public:
[adbsr-internal/satlas-downstream](https://huggingface.co/adbsr-internal/satlas-downstream).
The script that made this data:
[`build_demo2_package.py`](https://github.com/adb-sr/sr-client/blob/main/tools/demos/build_demo2_package.py).

---
Contains modified Copernicus Sentinel data. Building footprints: Google–Microsoft–OSM Open
Buildings by VIDA (ODbL). Roads: © OpenStreetMap contributors (ODbL), via Overture Maps.
OpenEarthMap tiles and labels: OpenEarthMap (Xia et al., 2023), CC BY-NC-SA 4.0.
Land-cover model: SatlasPretrain (Ai2) fine-tuned on OpenEarthMap; CC BY-NC-SA 4.0,
non-commercial use.